# Nemotron Reasoning — Colab A100 v4 (0.59 → 0.88+)

**Key changes over v3:**

| What changed | Why | Expected gain |
|---|---|---|
| Stage 1: Completion-only SFT (no packing) | Loss only on assistant tokens = stronger gradient signal | +5–10pp |
| Stage 2: **GRPO** on real train.csv | Ground-truth reward signal replaces teacher imitation | +15–25pp |
| Remove template CoTs from real data | Low-quality fake reasoning pollutes training | +2–5pp |
| A100-specific settings (larger batch, no quant on 80GB) | Better GPU utilization | +2–3pp efficient |
| Improved system prompt (explicit boxed-at-end instruction) | Format alignment with evaluation | +1–2pp |

**Expected timeline on A100 (40 GB, ~12-hr Colab):**
- Stage 1 SFT (synthetic only, completion-only): ~1–2 hr → submit (~0.65–0.72)
- Stage 2 GRPO (real train.csv, 500 rows, 2 epochs): ~3–5 hr → submit (~0.80–0.88)
- Stage 3 DPO (optional, needs OpenAI key): ~1 hr → submit (~0.82–0.90)

**No teacher API needed for Stages 1 and 2!** Teacher API only improves Stage 3 DPO.

**Recommended flow:** Run Stage 1 → submit → run Stage 2 → submit → stop or continue.

## Setup — upload project files

In [ ]:
import os, subprocess, sys, zipfile, getpass
from pathlib import Path
from google.colab import files

WORK_ROOT = Path('/content/project').resolve()
WORK_ROOT.mkdir(parents=True, exist_ok=True)

print('Upload udacity_upload.zip (your project archive) ...')
for name in files.upload():
    if name.endswith('.zip'):
        with zipfile.ZipFile(name) as zf:
            zf.extractall(WORK_ROOT)
        print('Extracted', name)

data_dir = WORK_ROOT / 'data'
(data_dir / 'reports').mkdir(parents=True, exist_ok=True)
(data_dir / 'synthetic').mkdir(parents=True, exist_ok=True)

if not (data_dir / 'train.csv').is_file():
    print('Upload train.csv ...')
    for n in files.upload():
        Path(n).rename(data_dir / n)

os.chdir(WORK_ROOT)
sys.path.insert(0, str(WORK_ROOT))
assert (WORK_ROOT / 'scripts' / '03_train_lora.py').is_file(), 'scripts not found — check zip'
print('cwd:', os.getcwd())

## Config

In [ ]:
import torch

MODEL_ID = 'nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16'
HF_TOKEN = os.environ.get('HF_TOKEN', '') or getpass.getpass('HF token (or blank if not needed): ')
# Teacher API (only needed for Stage 3 DPO — optional)
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', '') or ''
if OPENAI_API_KEY:
    os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {torch.cuda.get_device_name(0)} ({vram_gb:.0f} GB)')

# LoRA settings (competition max rank = 32)
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05

# --- Stage 1: SFT on synthetic data (warmup, no API needed) ---
# Completion-only is the key improvement: loss ONLY on assistant tokens
SFT_SYNTHETIC_PER_KIND = 2000   # 4 types × 2000 = 8000 examples
SFT_MAX_PER_TYPE = 2500
SFT_EPOCHS = 3.0
SFT_LR = 2e-4
SFT_BATCH = 2 if vram_gb < 60 else 4
SFT_GRAD_ACCUM = 8
SFT_MAX_SEQ = 3072              # shorter than v3 → fits better without packing

# --- Stage 2: GRPO on real train.csv (primary score booster) ---
GRPO_LIMIT = 500                # rows to use (0=all; start with 500 for A100 40GB)
GRPO_EPOCHS = 2
GRPO_LR = 5e-5
GRPO_BATCH = 1
GRPO_GRAD_ACCUM = 8
GRPO_NUM_GENERATIONS = 0        # 0 = auto (4 on 40GB, 8 on 80GB)
GRPO_MAX_NEW_TOKENS = 512       # shorter = faster; 1024 if accuracy plateaus
GRPO_TEMPERATURE = 0.8

# --- Stage 3: DPO refinement (optional, needs OpenAI key) ---
DPO_PAIRS_SAMPLES = 500
DPO_EPOCHS = 1.0
DPO_LR = 2e-6                   # very conservative — DPO is sensitive
COT_MODEL = 'gpt-4o-mini'       # cheap teacher for DPO pairs

print('Config set.')

## Install dependencies

In [ ]:
import re

def pip_install(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

def pip_try(*args):
    try:
        pip_install(*args)
        return True
    except subprocess.CalledProcessError:
        return False

pip_install('-U', 'pip', 'setuptools', 'wheel')
pip_install(
    'transformers>=4.45,<5',   # pin <5 avoids 4-bit OOM regression
    'peft>=0.12',
    'trl>=0.12',
    'datasets',
    'accelerate>=0.34',
    'bitsandbytes>=0.43',
    'psutil', 'pandas', 'numpy',
    'tqdm', 'huggingface_hub',
    'openai>=1.30',
    'ninja',
)

# mamba-ssm + causal-conv1d (required for Nemotron-3 Mamba layers)
_tf = torch.__version__
_mm = re.match(r'(\d+\.\d+)', _tf).group(1)
_cu = 'cu12' if 'cu12' in _tf else 'cu11'
_py = f'cp{sys.version_info.major}{sys.version_info.minor}'

for pkg, gh_base, tags in [
    ('causal-conv1d', 'https://github.com/Dao-AILab/causal-conv1d/releases/download',
     [('v1.6.1', 'causal_conv1d', '1.6.1'), ('v1.5.4', 'causal_conv1d', '1.5.4')]),
    ('mamba-ssm', 'https://github.com/state-spaces/mamba/releases/download',
     [('v2.3.1', 'mamba_ssm', '2.3.1'), ('v2.2.4', 'mamba_ssm', '2.2.4')]),
]:
    installed = False
    for tag, wn, wv in tags:
        for abi in ('cxx11abiTRUE', 'cxx11abiFALSE'):
            url = f'{gh_base}/{tag}/{wn}-{wv}+{_cu}torch{_mm}{abi}-{_py}-{_py}-linux_x86_64.whl'
            if pip_try(url):
                print(f'Installed {pkg} from wheel')
                installed = True
                break
        if installed:
            break
    if not installed:
        os.environ['CAUSAL_CONV1D_FORCE_BUILD'] = 'TRUE'
        os.environ['MAMBA_FORCE_BUILD'] = 'TRUE'
        pip_try('--no-build-isolation', '--no-deps', pkg)

# nvidia-cutlass-dsl for Nemotron custom ops
pip_try('--prefer-binary', 'nvidia-cutlass-dsl>=4.4', 'nvidia-cutlass-dsl-libs-base>=4.4')

print('All dependencies installed.')

## Download base model + auto-patch Nemotron dtype bug

In [ ]:
import glob, shutil
from huggingface_hub import login, snapshot_download

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)

MODEL_PATH = snapshot_download(MODEL_ID, resume_download=True)
print('Model at', MODEL_PATH)

# Patch modeling_nemotron_h.py: fix BFloat16/Float index_add_ mismatch in MoE layers
for mf in set(
    glob.glob('/root/.cache/huggingface/hub/**/modeling_nemotron_h.py', recursive=True)
    + glob.glob('/root/.cache/huggingface/modules/**/modeling_nemotron_h.py', recursive=True)
):
    lines = Path(mf).read_text().splitlines(True)
    changed = False
    for i, l in enumerate(lines):
        if ('final_hidden_states.index_add_(0, token_indices, weighted_output)' in l
                and 'weighted_output.to(' not in l):
            lines[i] = l.replace(
                'weighted_output)',
                'weighted_output.to(final_hidden_states.dtype))'
            )
            changed = True
        if '.to(expert_dtype)' in l:
            lines[i] = l.replace('.to(expert_dtype)', '.to(torch.bfloat16)')
            changed = True
    if changed:
        Path(mf).write_text(''.join(lines))
        pc = str(Path(mf).parent / '__pycache__')
        if Path(pc).exists():
            shutil.rmtree(pc)
        print('Patched', mf)

print('Model ready.')

## Stage 1 — SFT on synthetic data (format warmup, completion-only)

**What changed from v3:**
- `--completion-only` enabled: loss ONLY on assistant tokens (no system/user prompt tokens)
- No packing (incompatible with completion-only)
- No teacher API needed: synthetic data only
- No template CoTs for real train.csv (those are low quality; GRPO in Stage 2 handles real data)

Expected score after Stage 1: ~0.65–0.72

In [ ]:
# Build synthetic SFT dataset
# --skip-cot: no teacher API calls (synthetic data is self-contained)
# --skip-template-cot: skip weak template CoTs for real train.csv rows
#   (GRPO in Stage 2 handles real data with ground-truth reward signal)
cmd = [
    sys.executable, 'scripts/02_prepare_data.py',
    '--data-dir', 'data',
    '--synthetic-dir', 'data/synthetic',
    '--output', 'data/train_sft_v4.jsonl',
    '--tokenizer-model', str(MODEL_PATH),
    '--synthetic-per-kind', str(SFT_SYNTHETIC_PER_KIND),
    '--max-per-type', str(SFT_MAX_PER_TYPE),
    '--max-tokens-per-example', '5000',
    '--skip-cot',            # no teacher API calls
    '--skip-template-cot',   # skip weak template CoTs (GRPO handles real data)
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
env = os.environ.copy()
env['TOKENIZERS_PARALLELISM'] = 'false'
env['NEMOTRON_KAGGLE_PATCHES'] = '0'
env['PYTHONUNBUFFERED'] = '1'

# KEY CHANGE: --completion-only computes loss only on assistant tokens
# NO --packing (packing is incompatible with completion-only collation)
cmd = [
    sys.executable, 'scripts/03_train_lora.py',
    '--data-path', 'data/train_sft_v4.jsonl',
    '--output-dir', 'lora_adapter_sft',
    '--checkpoint-dir', 'lora_output_sft',
    '--model-path', str(MODEL_PATH),
    '--lora-target-mode', 'kaggle_nemotron',
    '--lora-r', str(LORA_R),
    '--lora-alpha', str(LORA_ALPHA),
    '--lora-dropout', str(LORA_DROPOUT),
    '--batch-size', str(SFT_BATCH),
    '--grad-accum', str(SFT_GRAD_ACCUM),
    '--epochs', str(SFT_EPOCHS),
    '--lr', str(SFT_LR),
    '--max-seq-length', str(SFT_MAX_SEQ),
    '--warmup-ratio', '0.05',
    '--max-grad-norm', '1.0',
    '--neftune-alpha', '5.0',
    '--completion-only',        # ← THE KEY IMPROVEMENT vs v3
    # NO --packing (incompatible with completion-only)
    '--force-peft',
    '--no-nemotron-kaggle-patches',
    '--dataloader-workers', '0',
]
print(' '.join(cmd), flush=True)
proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
rc = proc.wait()
if rc != 0:
    raise subprocess.CalledProcessError(rc, cmd)

### Package & submit Stage 1 adapter

In [ ]:
subprocess.run([
    sys.executable, 'scripts/05_package_submission.py',
    '--adapter-dir', 'lora_adapter_sft',
    '--output', 'submission_stage1_v4.zip',
], check=True)
from google.colab import files
files.download('submission_stage1_v4.zip')
print('Stage 1 done. Expected score ~0.65–0.72. Submit to Kaggle, then continue to Stage 2.')

## Stage 2 — GRPO on real train.csv (PRIMARY SCORE BOOSTER)

**Why GRPO beats SFT for this competition:**
- Train.csv has verified ground-truth answers → use them as reward signal
- Model explores diverse reasoning paths; GRPO reinforces what gets correct answers
- No teacher API needed (ground truth IS the teacher)
- Directly optimizes evaluation metric (\\boxed{} answer correctness)

**Expected score after Stage 2: ~0.80–0.88**

**Estimated runtime:** ~3–5 hr on A100 40GB (500 rows, 2 epochs, 4 rollouts)

In [ ]:
env2 = os.environ.copy()
env2['TOKENIZERS_PARALLELISM'] = 'false'
env2['PYTHONUNBUFFERED'] = '1'

cmd = [
    sys.executable, 'scripts/08_grpo.py',
    '--train-csv', 'data/train.csv',
    '--sft-adapter', 'lora_adapter_sft',   # warm-start from Stage 1
    '--base-model', str(MODEL_PATH),
    '--output-dir', 'lora_adapter_grpo',
    '--checkpoint-dir', 'grpo_output',
    '--lora-r', str(LORA_R),
    '--lora-alpha', str(LORA_ALPHA),
    '--lora-dropout', str(LORA_DROPOUT),
    '--epochs', str(GRPO_EPOCHS),
    '--lr', str(GRPO_LR),
    '--batch-size', str(GRPO_BATCH),
    '--grad-accum', str(GRPO_GRAD_ACCUM),
    '--num-generations', str(GRPO_NUM_GENERATIONS),
    '--max-new-tokens', str(GRPO_MAX_NEW_TOKENS),
    '--temperature', str(GRPO_TEMPERATURE),
    '--limit', str(GRPO_LIMIT),
    '--curriculum',   # start with shorter problems (faster early reward signal)
]
print(' '.join(cmd), flush=True)
proc = subprocess.Popen(cmd, env=env2, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
rc = proc.wait()
if rc != 0:
    raise subprocess.CalledProcessError(rc, cmd)

### Package & submit Stage 2 (GRPO) adapter

In [ ]:
subprocess.run([
    sys.executable, 'scripts/05_package_submission.py',
    '--adapter-dir', 'lora_adapter_grpo',
    '--output', 'submission_stage2_grpo.zip',
], check=True)
from google.colab import files
files.download('submission_stage2_grpo.zip')
print('Stage 2 (GRPO) done. Expected score ~0.80–0.88. Submit to Kaggle!')

## Stage 2b — GRPO Round 2: use all train.csv rows

If time and Colab session allow, run GRPO again on the full train.csv starting from the Stage 2 adapter.
Each additional GRPO epoch typically gives +1–3pp.

**Estimated extra time:** 2–4 hr for 1 more epoch on all rows.

In [ ]:
# Uncomment to run a second GRPO round on the full dataset
# cmd = [
#     sys.executable, 'scripts/08_grpo.py',
#     '--train-csv', 'data/train.csv',
#     '--sft-adapter', 'lora_adapter_grpo',  # warm-start from Stage 2
#     '--base-model', str(MODEL_PATH),
#     '--output-dir', 'lora_adapter_grpo2',
#     '--checkpoint-dir', 'grpo_output2',
#     '--lora-r', str(LORA_R),
#     '--lora-alpha', str(LORA_ALPHA),
#     '--epochs', '1',
#     '--lr', str(GRPO_LR * 0.5),   # lower LR for refinement
#     '--limit', '0',  # all rows
#     '--max-new-tokens', '768',   # slightly longer
#     '--curriculum',
# ]
# subprocess.run(cmd, env=env2, check=True)
print('Uncomment the block above to run GRPO round 2.')

## Stage 3 — DPO refinement (optional, needs OpenAI key)

Build preference pairs from teacher-verified CoTs vs. adapter-wrong answers, then DPO-tune.
Typical gain: +1–3pp on top of GRPO. **Only run if you have an OpenAI API key and time.**

In [ ]:
# Stage 3a: Build DPO preference pairs
if not OPENAI_API_KEY:
    print('OPENAI_API_KEY not set — skipping Stage 3. Set it above to enable DPO.')
else:
    cmd = [
        sys.executable, 'scripts/06b_build_dpo_pairs.py',
        '--adapter-path', 'lora_adapter_grpo',   # use GRPO adapter as starting point
        '--base-model', str(MODEL_PATH),
        '--train-csv', 'data/train.csv',
        '--output', 'data/dpo_pairs_v4.jsonl',
        '--max-samples', str(DPO_PAIRS_SAMPLES),
        '--cot-backend', 'openai',
        '--cot-model', COT_MODEL,
    ]
    print(' '.join(cmd))
    subprocess.run(cmd, check=True)

In [ ]:
# Stage 3b: DPO training
if not OPENAI_API_KEY or not Path('data/dpo_pairs_v4.jsonl').is_file():
    print('Skipping DPO training (no pairs file or no API key).')
else:
    cmd = [
        sys.executable, 'scripts/07_dpo.py',
        '--pairs', 'data/dpo_pairs_v4.jsonl',
        '--sft-adapter', 'lora_adapter_grpo',   # start from GRPO adapter
        '--base-model', str(MODEL_PATH),
        '--output-dir', 'lora_adapter_dpo',
        '--checkpoint-dir', 'dpo_output',
        '--epochs', str(DPO_EPOCHS),
        '--lr', str(DPO_LR),
        '--batch-size', '1',
        '--grad-accum', '8',
        '--beta', '0.05',     # lower beta for more plasticity post-GRPO
        '--max-length', '3072',
    ]
    print(' '.join(cmd))
    proc = subprocess.Popen(cmd, env=env2, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='', flush=True)
    if proc.wait():
        raise SystemExit('DPO failed')
    
    subprocess.run([
        sys.executable, 'scripts/05_package_submission.py',
        '--adapter-dir', 'lora_adapter_dpo',
        '--output', 'submission_stage3_dpo.zip',
    ], check=True)
    from google.colab import files
    files.download('submission_stage3_dpo.zip')
    print('Stage 3 DPO done. Expected score ~0.82–0.90. Submit!')

## If GRPO runs too slow: SFT+Teacher CoT fallback

If the A100 session ends before GRPO finishes, use teacher CoT for all train.csv rows instead.
This uses gpt-4o-mini (~$2 for all rows) and produces good SFT signal with completion-only training.

In [ ]:
# Fallback: SFT with teacher CoTs for all train.csv rows (needs OPENAI_API_KEY)
# Run this INSTEAD of GRPO if session time is too short for GRPO

# Uncomment to use:
# if not OPENAI_API_KEY:
#     print('Set OPENAI_API_KEY first.')
# else:
#     cmd = [
#         sys.executable, 'scripts/02_prepare_data.py',
#         '--data-dir', 'data',
#         '--synthetic-dir', 'data/synthetic',
#         '--output', 'data/train_sft_v4_with_cot.jsonl',
#         '--tokenizer-model', str(MODEL_PATH),
#         '--synthetic-per-kind', str(SFT_SYNTHETIC_PER_KIND),
#         '--max-per-type', str(SFT_MAX_PER_TYPE),
#         '--max-tokens-per-example', '5000',
#         '--cot-backend', 'openai',
#         '--cot-model', 'gpt-4o-mini',  # cheap; ~$2 for all rows
#         '--cot-max-tokens', '2048',
#     ]
#     subprocess.run(cmd, check=True)
#     # Then run SFT with completion-only on this enriched dataset

print('Fallback SFT block is commented out. Uncomment and run if GRPO is too slow.')

## Notes & Troubleshooting

**Why does completion-only matter so much?**  
Without it, the model optimizes for predicting the system prompt + user prompt tokens (which are fixed).  
With it, every gradient step teaches the model to generate better assistant responses.  
On a typical run, completion-only gives the same final loss with 3–5× fewer effective gradient steps.

**Why GRPO over more SFT epochs?**  
SFT teaches the model to imitate (possibly wrong) teacher CoTs.  
GRPO teaches the model to get the RIGHT answer, regardless of CoT style.  
The competition only scores answer correctness, not CoT quality.

**GRPO out of memory?**  
- Reduce `GRPO_NUM_GENERATIONS` to 2
- Reduce `GRPO_MAX_NEW_TOKENS` to 256
- Reduce `GRPO_LIMIT` to 200

**GRPO reward stays near 0 after 50 steps?**  
- The SFT warm-start (Stage 1) helps. Make sure `--sft-adapter lora_adapter_sft` is set.
- Increase `GRPO_TEMPERATURE` to 0.9 for more diversity in rollouts.
- If still stuck: run a few more SFT epochs on synthetic data first.

**DPO destabilizes (eval loss spikes)?**  
- Lower `DPO_LR` to 5e-7.
- The GRPO adapter is already well-trained; DPO is fine-tuning on the margin.
- Skip DPO and submit GRPO adapter directly.

**Submission format reminder:**  
The `adapter_config.json` must have `base_model_name_or_path` set to the full Nemotron model ID.  
The packaging script (`05_package_submission.py`) handles this automatically.